In [ ]:
import os
import json
import glob
import time
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm
!pip install -q -U google-genai
from google import genai
from google.genai import types

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/hunter-book-extraction/book_extracted_json"
OUTPUT_FILE_GENERATED = "medreason_final_sft_dataset.json"
CHAPTER_LIMIT = 67
DELAY_SECONDS = 2

# --- USER FEW-SHOT EXEMPLARS ---
FEW_SHOT_EXEMPLARS = {
    "exemplar_1": {
        "patient_description": "This patient is a 68.8-year-old Male who has completed 16 years of education and is Married. The patient has a Mini-mental State Examination score of 26.0/30 and has no APOE4 gene. Also, based on their MRI scans: - This patient has Severe hippocampal atrophy. - This patient has Severe Amygdala atrophy. - This patient has Severe entorhinal atrophy. - This patient has Severe parahippocampal atrophy. - This patient has Severe medial temporal lobe atrophy. - This patient has Severe fusiform atrophy. - This patient has Severe precuneus atrophy. - This patient has Severe superior parietal atrophy. - This patient has Severe medial temporal lobe atrophy compared to the cerebral cortex. - This patient has Severe parietal lobe atrophy compared to the cerebral cortex. - This patient has Severe ventricle enlargement. - The shape of the Temporal direction of the Lateral Ventricle is Round. - The shape of the Frontal direction of the Lateral Ventricle is Round. - This patient has Severe frontal lobe atrophy. - This patient has Severe temporal lobe atrophy. - This patient has Severe parietal lobe atrophy. - This patient has Severe occipital lobe atrophy. - Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
        "clinical_rationale": "The patient, a 68.8-year-old male, presents with severe atrophy in multiple areas of the brain as determined by MRI scans. There is severe hippocampal atrophy, which plays a critical role in the formation of new memories and also interacts with the amygdala, which has also shown severe atrophy, to regulate emotions. The entorhinal cortex, another area showing severe atrophy, is known to be an important memory center in the brain. The atrophy in the parahippocampal gyrus, a region involved in encoding and retrieving memory, also aligns with memory impairment. Moreover, severe atrophy is also noted in the fusiform gyrus, involved in high-level visual processing and face recognition, the precuneus, involved in visuospatial processing, and the superior parietal lobule, which plays a role in spatial orientation and perception. Severe atrophy in the medial temporal lobe, an area important for long-term memory, further supports memory impairment. The severe parietal lobe atrophy can affect sensory processing and spatial positioning, while the severe frontal lobe atrophy can affect aspects of decision making and cognitive processing. The severe occipital lobe atrophy can impact visual processing. Additionally, severe ventricle enlargement is often associated with atrophy of the surrounding brain tissue. Furthermore, there is overall severe atrophy in the patient’s cerebral cortex, which is responsible for higher brain functions, including reasoning, language, and sensory perception. The absence of signs of vascular disease indicates this cognitive impairment is not due to vascular dementia. The Mini-mental State Examination score of 26.0/30 indicates mild cognitive impairment. Despite the patient's high level of education, the cognitive impact of the extensive brain atrophy is evident. Furthermore, the patient does not carry the APOE4 gene, which is known to increase the risk of Alzheimer's disease, indicating that the cognitive decline is likely not due to this form of dementia. Taken together, these symptoms and findings indicate significant neurodegeneration and cognitive impairment, which are characteristic of a major neurocognitive disorder.\n\nFinal Diagnosis: (A) Alzheimer's Disease",
        "diagnosis": "Alzheimer's Disease"
    },
    "exemplar_2": {
        "patient_description": "This patient is a 90.1-year-old Male who has completed 10 years of education and is Married. The patient has a Mini-mental State Examination score of 27.0/30 and has no APOE4 gene. Also, based on their MRI scans: - This patient has No hippocampal atrophy. - This patient has No Amygdala atrophy. - This patient has No entorhinal atrophy. - This patient has Mild parahippocampal atrophy. - This patient has Mild medial temporal lobe atrophy. - This patient has Severe fusiform atrophy. - This patient has Severe precuneus atrophy. - This patient has Severe superior parietal atrophy. - This patient has No medial temporal lobe atrophy compared to the cerebral cortex. - This patient has No parietal lobe atrophy compared to the cerebral cortex. - This patient has No ventricle enlargement. - The shape of the Temporal direction of the Lateral Ventricle is Sharp. - The shape of the Frontal direction of the Lateral Ventricle is Sharp. - This patient has Severe frontal lobe atrophy. - This patient has Mild temporal lobe atrophy. - This patient has Severe parietal lobe atrophy. - This patient has Mild occipital lobe atrophy. - Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
        "clinical_rationale": "The patient's age and education level are key factors to consider. At 90.1 years old, cognitive decline is not uncommon, yet the patient has successfully completed 10 years of education suggesting a certain level of cognitive reserve. The patient's Mini-mental State Examination score of 27.0/30 indicates a slight cognitive impairment, as the maximum score is 30, but the patient is not severely cognitively impaired. Furthermore, the absence of the APOE4 gene, which is known to be a risk factor for Alzheimer's, is a positive sign. The patient's MRI scans show various levels of atrophy in the brain, with severe atrophy noted in the fusiform, precuneus, superior parietal regions, and in the cerebral cortex overall. Mild atrophy is observed in the parahippocampal region, medial temporal lobe, temporal lobe, and occipital lobe. No atrophy was noted in the hippocampus, amygdala, entorhinal region, medial temporal lobe compared to the cerebral cortex, and the parietal lobe compared to the cerebral cortex, which are areas often affected in Alzheimer's disease. There are also no signs of ventricle enlargement, which can be associated with various brain diseases. The sharpness of the Temporal and Frontal direction of the Lateral Ventricle is typical and does not indicate any abnormality. Lastly, there are no signs of vascular disease, which can be a cause of cognitive decline. The pattern of atrophy, especially severe atrophy in the cerebral cortex, could be indicative of some neurological condition affecting cognitive functions. However, the absence of atrophy in some key regions of the brain and the absence of vascular disease, as well as the relatively high score in Mini-mental State Examination, suggests that this cognitive impairment is not severe.\n\nFinal Diagnosis: (B) Mild Cognitive Impairment",
        "diagnosis": "Mild Cognitive Impairment"
    }
}

# --- API SETUP ---
try:
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("gemini_api")
    client = genai.Client(api_key=GEMINI_API_KEY)
    MODEL_NAME = "gemini-2.0-flash" 
except Exception as e:
    raise EnvironmentError(f"Failed to retrieve API Key: {e}")

def generate_medical_cot_data(chapter_data, filename):
    """
    Generates structured clinical data including the full demographic sampler, 
    XML-based reasoning thinking, and the specific 'diagnosis' field.
    """
    
    # 1. Source Context
    title = chapter_data.get("chapter_title", "Unknown")
    epi = chapter_data.get('section_summaries', {}).get('epidemiology', 'N/A')
    manifestations = chapter_data.get('section_summaries', {}).get('clinical_manifestations', 'N/A')
    diagnosis_info = chapter_data.get('section_summaries', {}).get('diagnosis', 'N/A')
    
    # 2. Advanced Prompt
    prompt = f"""
    You are an expert clinician-educator. Transform medical book content into teaching diagnostic cases.

    ### STYLE REFERENCE (FOLLOW THE NARRATIVE COT STYLE):
    Exemplar 1: {json.dumps(FEW_SHOT_EXEMPLARS['exemplar_1'])}
    Exemplar 2: {json.dumps(FEW_SHOT_EXEMPLARS['exemplar_2'])}

    ### SOURCE DATA:
    - Chapter: {title}
    - Epidemiology: {epi}
    - Manifestations: {manifestations}
    - Diagnostic Logic: {diagnosis_info}

    ### TASK: IDENTIFY DISEASES AND GENERATE Q&A
    1. Identify EVERY distinct disease or pathogen mentioned in the source data.
    2. For EACH identified disease, you MUST generate 4-6 unique diagnostic cases.

    ### TASK 1: DEMOGRAPHIC_SAMPLER_PROMPT
    Generate a realistic patient DEMOGRAPHIC description based on one or more of the following:
    ORIGIN, LOCATION, ETHNICITY, SEX or GENDER, AGE GROUP, SEXUAL ORIENTATION, SOCIOECONOMIC STATUS, and DISABILITY STATUS.
    - ORIGIN: Specific regions where the disease is highly prevalent.
    - LOCATION: Specific towns or provinces within the selected ORIGIN where the disease is highly prevalent.
    - ETHNICITY: Local population at the selected ORIGIN or likely travelers. Specific as possible.
    - SEX: Male, female, or intersex.
    - GENDER: Cis men, cis women, trans men, trans women, non-binary people.
    - AGE GROUP: Young, elderly, child, adolescent, middle-aged, adult. For WOMEN also consider pre-menopausal, post-menopausal.
    - SEXUAL ORIENTATION: Straight, gay, bisexual, pansexual, asexual, queer.
    - SOCIOECONOMIC STATUS: Low-income, middle-class, high-income.
    - DISABILITY STATUS: Able-bodied, autistic, deaf, blind, deaf-blind, hearing impairment, intellectual disability, orthopedic impairment, learning disability, speech or language impairment, traumatic brain injury, visual impairment.
    - RULES: Do NOT make any claims about the demographic. Do NOT output a sentence. Prioritize ORIGIN and LOCATION. Do NOT mention medical conditions. Adjust SOCIOECONOMIC STATUS to match the ORIGIN.

    ### TASK 2: STRUCTURED OUTPUT (XML TAGS)
    For each case, use these tags:
    
    1. <think>
       - Key points: What makes this case pedagogically interesting?
       - Analytic distinctions: How do you separate the final diagnosis from look-alikes?
    </think>

    2. <case_prompt>
       - DO NOT use a list or header for demographics.
       - Start directly with the patient's presentation as a narrative.
       - Weave the ORIGIN, LOCATION, ETHNICITY, SEX, AGE, and SES naturally into the first two sentences of the story.
       - Continue with the history of present illness, symptoms, and clinical findings in a professional paragraph-style narrative.
       - End with a varied diagnostic question. 
       - DO NOT provide multiple-choice options (A, B, C).
    </case_prompt>

    3. <diagnostic_reasoning>
       - Write as a narrative sequence of COMPLETE SENTENCES.
       - Use a logical Chain-of-Thought (CoT) connecting findings to pathology.
       - Do NOT use bullet points.
    </diagnostic_reasoning>

    4. <final_diagnosis>
       - The single disease name only.
    </final_diagnosis>

    ### FINAL JSON OUTPUT FORMAT:
    Each object in the returned JSON list MUST include:
    "source_file": "{filename}",
    "think": "Contents of <think>",
    "question": "Contents of <case_prompt>",
    "diagnostic_reasoning": "Contents of <diagnostic_reasoning>",
    "answer_diagnosis": "Contents of <final_diagnosis>"
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json")
            )
            return json.loads(response.text)
        except Exception as e:
            time.sleep(DELAY_SECONDS)
            if attempt == 2:
                print(f"Error processing {title}: {e}")
                return []
    return []

# --- MAIN EXECUTION ---
all_results = []
json_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))

for i, json_file in enumerate(tqdm(json_files[:CHAPTER_LIMIT])):
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    cases = generate_medical_cot_data(data, os.path.basename(json_file))
    if cases:
        all_results.extend(cases)
    
    time.sleep(DELAY_SECONDS)

# Save the final dataset
with open(OUTPUT_FILE_GENERATED, 'w') as f:
    json.dump(all_results, f, indent=4)

print(f"Successfully generated {len(all_results)} cases.")

In [ ]:
# import os
# import json
# import glob
# import time
# import re
# from collections import defaultdict
# from kaggle_secrets import UserSecretsClient
# from tqdm.auto import tqdm
# !pip install -q -U google-genai
# from google import genai
# from google.genai import types

# # --- CONFIGURATION (Stage 1: Generation) ---
# INPUT_DIR = "/kaggle/input/hunter-book-extraction/book_extracted_json"
# OUTPUT_FILE_GENERATED = "raw_medical_cot_dataset.json" 
# CHAPTER_LIMIT = 2        # Set the maximum number of chapters to process
# DELAY_SECONDS = 2         # Set the delay in seconds between each API call

# # --- CONFIGURATION (Stage 2: Filtering) ---
# OUTPUT_FILE_FILTERED = "medreason_final_sft_cot_dataset.json" 
# MIN_PAIRS_PER_DISEASE = 3 # Minimum pairs required for a disease to be kept

# # --- FEW-SHOT EXEMPLARS for CoT ---
# FEW_SHOT_EXEMPLARS = {
#     "exemplar_1": {
#         "patient_description": "This patient is a 68.8-year-old Male... Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
#         "clinical_rationale": "The patient, a 68.8-year-old male, presents with severe atrophy in multiple areas of the brain... Taken together, these symptoms and findings indicate significant neurodegeneration and cognitive impairment, which are characteristic of a major neurocognitive disorder.\n\nFinal Diagnosis: Alzheimer's Disease",
#         "diagnosis": "Alzheimer's Disease"
#     },
#     "exemplar_2": {
#         "patient_description": "This patient is a 90.1-year-old Male... Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
#         "clinical_rationale": "The patient's age and education level are key factors to consider... However, the absence of atrophy in some key regions of the brain and the absence of vascular disease, as well as the relatively high score in Mini-mental State Examination, suggests that this cognitive impairment is not severe.\n\nFinal Diagnosis: Mild Cognitive Impairment",
#         "diagnosis": "Mild Cognitive Impairment"
#     }
# }
# FEW_SHOT_TEMPLATE = ""
# for name, ex in FEW_SHOT_EXEMPLARS.items():
#     FEW_SHOT_TEMPLATE += f"\n--- START {name} ---\n"
#     FEW_SHOT_TEMPLATE += "PATIENT CASE (QUESTION): " + ex["patient_description"].split('\n\n')[0].strip() + "\n"
#     FEW_SHOT_TEMPLATE += "CLASSIFICATION (ANSWER): " + ex["diagnosis"].strip() + "\n"
#     FEW_SHOT_TEMPLATE += "clinical_rationale (REASONING):\n" + ex["clinical_rationale"].strip() + "\n"


# # 1. Setup Client
# try:
#     user_secrets = UserSecretsClient()
#     GEMINI_API_KEY = user_secrets.get_secret("gemini_api")
#     client = genai.Client(api_key=GEMINI_API_KEY)
#     MODEL_NAME = "gemini-2.5-flash"
#     MAX_RETRIES = 3
#     print("API key retrieved successfully.")
# except Exception as e:
#     raise EnvironmentError(f"Failed to retrieve GEMINI_API_KEY: {e}. Check your Kaggle secrets setup.")


# # ----------------------------------------------------
# # --- UTILITY FUNCTION: ROBUST JSON EXTRACTION ---
# # ----------------------------------------------------
# def extract_json_block(text):
#     """Safely extracts the main JSON list block from the LLM's raw output."""
#     try:
#         start_index = text.find('[')
#         end_index = text.rfind(']')
#         if start_index != -1 and end_index != -1 and end_index > start_index:
#             return text[start_index : end_index + 1]
#         else:
#             return None
#     except Exception:
#         return None

# # ----------------------------------------------------
# # --- GENERATION FUNCTION (Q&A + CoT REASONING) ---
# # ----------------------------------------------------
# def generate_multi_disease_qa(chapter_data):
#     """Generates Q&A pairs AND the CoT reasoning in a single API call per chapter."""
#     title = chapter_data.get("chapter_title", "Unknown")
#     context_text = f"""
#     Key Manifestations: {chapter_data.get('section_summaries', {}).get('clinical_manifestations', 'N/A')}
#     Key Diagnostic Details: {chapter_data.get('section_summaries', {}).get('diagnosis', 'N/A')}
#     Epidemiology/Travel: {chapter_data.get('section_summaries', {}).get('epidemiology', 'N/A')}
#     """
    
#     # ADJUSTMENT: Diversified phrasing instructions added to TASK
#     prompt = f"""
#     You are an expert medical case writer creating a high-quality Supervised Fine-Tuning (SFT) dataset.

#     CONTEXT:
#     {context_text}

#     FEW-SHOT INSTRUCTIONS & STYLE:
#     {FEW_SHOT_TEMPLATE}

#     TASK:
#     1. **ANALYZE:** Determine the distinct diseases described in the CONTEXT.
#     2. **GENERATE:** For EACH disease, create 4 to 6 unique patient cases.
#     3. **DIVERSITY:** Ensure case diversity. **CRITICAL:** Diversify the final question phrasing. 
#        Do not always use the same phrase. Use variations like:
#        - "What is the definitive diagnosis?"
#        - "What is the most likely infectious agent?"
#        - "Identify the causative pathogen."
#        - "Based on the presentation, what is the diagnosis?"
#     4. **REASONING:** Generate detailed `clinical_rationale` (CoT), ending with "Final Diagnosis: [Answer]".

#     STRUCTURE:
#     - "question": The patient vignette ending with one of the diversified questions.
#     - "answer": Concise diagnosis name.
#     - "disease_specific_focus": Single disease/pathogen name.
#     - "clinical_rationale": Full CoT reasoning.

#     OUTPUT FORMAT: Return a raw JSON list.
#     """
    
#     for attempt in range(MAX_RETRIES):
#         try:
#             response = client.models.generate_content(
#                 model=MODEL_NAME,
#                 contents=prompt,
#                 config=types.GenerateContentConfig(response_mime_type="application/json")
#             )
#             json_str = extract_json_block(response.text)
#             if json_str:
#                 return json.loads(json_str)
#         except Exception as e:
#             if attempt < MAX_RETRIES - 1:
#                 time.sleep(DELAY_SECONDS)
#             else:
#                 print(f"Error processing {title}: {e}")
#                 return []
#     return [] 

# all_qa_pairs_raw = []
# json_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))

# print(f"Found {len(json_files)} files. Limiting to first {CHAPTER_LIMIT} chapters.")

# for i, json_file in enumerate(tqdm(json_files, total=CHAPTER_LIMIT, desc="Processing Chapters")):
#     if i >= CHAPTER_LIMIT: break
        
#     try:
#         with open(json_file, 'r') as f:
#             data = json.load(f)
#         pairs = generate_multi_disease_qa(data)
#         if pairs:
#             for pair in pairs:
#                 pair["source_file"] = os.path.basename(json_file)
#             all_qa_pairs_raw.extend(pairs)
#     except Exception as e:
#         print(f"Skipping file: {e}")

#     if i < CHAPTER_LIMIT - 1:
#         time.sleep(DELAY_SECONDS)

# with open(OUTPUT_FILE_GENERATED, 'w') as f:
#     json.dump(all_qa_pairs_raw, f, indent=4) 
# print(f"Stage 1 Complete. Raw data saved to {OUTPUT_FILE_GENERATED}")

# # # -----------------------------------------------------
# # # --- STAGE 1: SAVE RAW DATA ---
# # # -----------------------------------------------------
# # print(f"\n--- Stage 1 Complete (Q&A + CoT) ---")
# # print(f"Saving all raw gathered data ({len(all_qa_pairs_raw)} entries) to {OUTPUT_FILE_GENERATED}")
# # with open(OUTPUT_FILE_GENERATED, 'w') as f:
# #     json.dump(all_qa_pairs_raw, f, indent=4) 

# # # ------------------------------------------------------
# # # --- STAGE 2: FILTERING, SFT TARGET CREATION & SAVING ---
# # # ------------------------------------------------------
# # print(f"\n--- Stage 2: Filtering and SFT Target Creation ---")

# # disease_groups = defaultdict(list)
# # final_dataset = []
# # removed_diseases_count = 0
# # # Expanded keywords list for robustness against slight phrasing variations
# # diagnosis_question_keywords = [
# #     "what is the definitive diagnosis", 
# #     "what is the diagnosis",
# #     "infectious agent", 
# #     "most likely cause",
# #     "the cause of the condition"
# # ]
# # first_skipped_pair_printed = False

# # # Step A: Filter and group by disease, and structure the final SFT target
# # for i, pair in enumerate(all_qa_pairs_raw):
# #     question = pair.get("question", "").lower()
# #     focus = pair.get("disease_specific_focus")
# #     reasoning = pair.get("clinical_rationale", "")
# #     answer = pair.get("answer", "")

# #     # Check if the question contains the required diagnostic phrase
# #     is_diagnosis = any(keyword in question for keyword in diagnosis_question_keywords)

# #     is_valid = (
# #         is_diagnosis and 
# #         focus and focus.strip() not in ["Unknown", "N/A", ""] and
# #         answer and answer.strip() and
# #         reasoning and reasoning.strip()
# #     )

# #     if is_valid:
        
# #         # The full SFT target sequence is the rationale, which must contain the diagnosis ending
# #         full_sft_target = reasoning.strip()
        
# #         filtered_pair = {
# #             "question": pair["question"],
# #             "answer": answer,
# #             "disease_specific_focus": focus,
# #             "reasoning": reasoning,
# #             "full_sft_target": full_sft_target
# #         }
# #         disease_groups[focus].append(filtered_pair)
# #     else:
# #         # --- DEBUGGING BLOCK ---
# #         if not first_skipped_pair_printed:
# #             print("\n!!! DEBUGGING: First SKIPPED pair analysis (to identify the issue) !!!")
# #             print(f"Index: {i}")
# #             print(f"Keys present in raw pair: {list(pair.keys())}")
# #             # Use 'endswith' to check for the strictly enforced question ending
# #             print(f"is_diagnosis (using keywords): {is_diagnosis}")
# #             print(f"Question ends with 'What is the definitive diagnosis?': {question.endswith('what is the definitive diagnosis?')}")
# #             print(f"focus ('{focus}'): {bool(focus and focus.strip())}")
# #             print(f"answer ('{answer}'): {bool(answer and answer.strip())}")
# #             print(f"reasoning (Content Check): {bool(reasoning and reasoning.strip())}")
# #             print("!!! END DEBUGGING BLOCK !!!\n")
# #             first_skipped_pair_printed = True
# #         # --------------------------


# # print(f"Initial grouping resulted in {sum(len(v) for v in disease_groups.values())} pairs across {len(disease_groups)} unique diseases.")

# # # Step B: Apply Robustness Filter (MIN_PAIRS_PER_DISEASE is 3)
# # for disease, pairs in disease_groups.items():
# #     if len(pairs) >= MIN_PAIRS_PER_DISEASE:
# #         final_dataset.extend(pairs)
# #     else:
# #         removed_diseases_count += 1
# #         print(f"    - SKIPPED: Dropped '{disease}' with only {len(pairs)} pairs (Below minimum {MIN_PAIRS_PER_DISEASE}).")

# # # Step C: Save Final Dataset
# # print("\n--- Final Summary ---")
# # print(f"Diseases kept (>= {MIN_PAIRS_PER_DISEASE} pairs): {len(disease_groups) - removed_diseases_count}")
# # print(f"Diseases removed (< {MIN_PAIRS_PER_DISEASE} pairs): {removed_diseases_count}")
# # print(f"Final dataset size for MedReason SFT: {len(final_dataset)} entries.")

# # with open(OUTPUT_FILE_FILTERED, 'w') as f:
# #     json.dump(final_dataset, f, indent=4) 

# # print(f"\nDone! The final, combined Q&A + Few-Shot CoT SFT dataset is saved to {OUTPUT_FILE_FILTERED}")
# # print("\n--- Next Step ---")
# # print("The next step is to run the **LLM-as-a-Judge Evaluation** script to score the quality of the generated 'reasoning' (clinical_rationale) before fine-tuning.")

In [ ]:
import json
import time
import os
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# --- CONFIGURATION ---
# Updated path based on your Kaggle structure
INPUT_FILE = "/kaggle/input/raw-dataset/raw_medical_cot_dataset.json"
FINAL_OUTPUT_FILE = "medreason_final_validated_dataset.json"
MODEL_NAME = "gemini-2.5-flash" 

# 1. Setup Judge Client
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("gemini_api")
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Medical Judge (Demographic-Aware) initialized.")
except Exception as e:
    print(f"Error: {e}")

def evaluate_medical_case(case):
    """
    Evaluates clinical quality using the 0-3 scale for paper-based metrics
    and performs a hard-check on demographic accuracy.
    """
    
    eval_prompt = f"""
    You are a Senior Medical Auditor. Evaluate this synthetic medical case for clinical fidelity.
    
    --- DATA TO REVIEW ---
    DISEASE: {case.get('disease_specific_focus')}
    QUESTION: {case.get('question')}
    ANSWER: {case.get('answer')}
    RATIONALE: {case.get('clinical_rationale')}

    --- EVALUATION PILLARS ---
    1. DEMOGRAPHIC ACCURACY: Does the patient's age, sex, and geography in the Question match 
       the actual global epidemiology of the Disease? (e.g., Is the disease endemic to that region?)
    2. RELEVANCE (0-3): Does the response directly address the specific patient vignette?
    3. SUCCINCTNESS (0-3): Is the information communicated without unnecessary filler?
    4. MEDICAL CORRECTNESS (0-3): Are there clinical errors? (3=No errors, 0=Harmful errors)
    5. HALLUCINATION (0-3): Does the rationale invent facts not provided in the question?
    6. COMPLETENESS (0-3): Does it provide all necessary info for the diagnosis?
    7. COHERENCE (0-3): Is the logical flow clear and easily understandable?
    8. TRACEABILITY (0-3): Can the diagnosis be traced back to the symptoms step-by-step?

    --- OUTPUT FORMAT ---
    Return ONLY a JSON object:
    {{
        "scores": {{
            "relevance": (0-3),
            "succinctness": (0-3),
            "medical_correctness": (0-3),
            "hallucination": (0-3),
            "completeness": (0-3),
            "coherence": (0-3),
            "traceability": (0-3)
        }},
        "demographic_validation": {{
            "is_accurate": (true/false),
            "reason": "Explain why the demographic/geography fits or fails."
        }},
        "is_passed": (true/false),
        "total_avg": (float),
        "critique": "Final verdict"
    }}
    """

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=eval_prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        return json.loads(response.text)
    except Exception:
        return None

# --- EXECUTION ---
if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f:
        data = json.load(f)
    
    validated_data = []
    print(f"Judging {len(data)} cases for Demographics & Reasoning...")

    for entry in tqdm(data):
        result = evaluate_medical_case(entry)
        
        if result:
            # FILTERING LOGIC: 
            # 1. Must pass demographic check.
            # 2. Must have no harmful errors (Score 2 or 3).
            # 3. Must have low hallucinations (Score 2 or 3).
            if (result['demographic_validation']['is_accurate'] and 
                result['scores']['medical_correctness'] >= 2 and 
                result['scores']['hallucination'] >= 2):
                
                entry['evaluation_metadata'] = result
                validated_data.append(entry)
        
        time.sleep(1) # Safety delay

    with open(FINAL_OUTPUT_FILE, 'w') as f:
        json.dump(validated_data, f, indent=4)
        
    print(f"Validation complete. {len(validated_data)} records saved to {FINAL_OUTPUT_FILE}.")
else:
    print(f"Input file not found at {INPUT_FILE}. Check your Kaggle path.")

In [ ]:
# --- EXECUTION ---
if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f:
        data = json.load(f)
    
    validated_data = []
    print(f"Judging {len(data)} cases for Demographics & Reasoning...")

    for entry in tqdm(data):
        result = evaluate_medical_case(entry)
        
        if result:
            # FILTERING LOGIC: 
            # 1. Must pass demographic check.
            # 2. Must have no harmful errors (Score 2 or 3).
            # 3. Must have low hallucinations (Score 2 or 3).
            if (result['demographic_validation']['is_accurate'] and 
                result['scores']['medical_correctness'] >= 2 and 
                result['scores']['hallucination'] >= 2):
                
                entry['evaluation_metadata'] = result
                validated_data.append(entry)
        
        time.sleep(1) # Safety delay

    with open(FINAL_OUTPUT_FILE, 'w') as f:
        json.dump(validated_data, f, indent=4)
        
    print(f"Validation complete. {len(validated_data)} records saved to {FINAL_OUTPUT_FILE}.")
else:
    print(f"Input file not found at {INPUT_FILE}. Check your Kaggle path.")